In [ ]:
import sys
sys.path.insert(0, "../")
import warnings
import pystac
import fsspec
import cartopy
import math
import calendar
import pandas as pd

import xarray as xr
import zarr
import matplotlib.pyplot as plt
from pprint import pprint
import json

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import copernicusmarine
from reefugia import fetch
from reefugia.constants import KELVIN_TO_CELSIUS
from reefugia.constants import BASELINE_START, BASELINE_END
from reefugia.constants import ANALYSIS_START, ANALYSIS_END
from reefugia.fetch import REGIONS

warnings.simplefilter("ignore", RuntimeWarning)

In [ ]:
print(json.dumps(REGIONS, indent=2))
print(BASELINE_START, BASELINE_END)
print(ANALYSIS_START, ANALYSIS_END)

In [ ]:
ds = fetch.fetch_data("red_sea", "2023-01-01", "2023-12-01")
ds["analysed_sst"]

In [ ]:
avg_sst_overspace = ds["analysed_sst"].sel(time="2023-01-01").mean(
    dim=["latitude", "longitude"]).compute()
print (avg_sst_overspace.values + KELVIN_TO_CELSIUS)

In [ ]:
avg_sst_overtime = ds["analysed_sst"].mean(dim="time")
sst_celsius = avg_sst_overtime.compute() + KELVIN_TO_CELSIUS
vmin = math.floor(float(sst_celsius.min().values))
vmax = math.ceil(float(sst_celsius.max().values))

In [ ]:
sst_celsius.attrs["long_name"] = "Sea Surface Temperature"
sst_celsius.attrs["units"] = "\u00b0C"
pprint(sst_celsius.attrs)

# Projections:
* PlateCarree — simple rectangular lat/lon grid. Good for regional maps where distortion isn't a big issue. Easy to work with because coordinates map directly to x/y.
- Mercator — preserves shapes but distorts size (Greenland looks as big as Africa). Good for navigation, bad for showing area accurately.
* LambertConformal — preserves shapes locally, good for mid-latitude regional maps (e.g. USA, Europe). Commonly used in weather forecasting.
* Mollweide — equal area, good for global maps showing distributions. Oval shape.
* Orthographic — looks like a globe viewed from space. Good for showing a hemisphere dramatically.
* Robinson — compromise projection, commonly used for world maps in atlases.

In [ ]:
# Use xarray's .plot() for quick exploration plots, to see how the data looks.
# One line, automatic labels, good enough for checking your data looks right.

fig, ax = plt.subplots(
    figsize=(10,6), dpi=100, 
    subplot_kw=dict(projection = ccrs.PlateCarree()))
sst_celsius.plot(cmap="RdYlBu_r",
                vmin=vmin,
                vmax=vmax)
ax.coastlines()

In [ ]:
# ax.pcolormesh() with cartopy — better for final outputs. 
# For proper map projections, coastlines, gridlines, 
# full control over every visual element. 

fig, ax = plt.subplots(
    figsize=(10,6), dpi=100, subplot_kw={"projection": 
                                         ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND)
mesh = ax.pcolormesh(sst_celsius.longitude, sst_celsius.latitude,
              sst_celsius.values, transform=ccrs.PlateCarree(),
             cmap="RdYlBu_r", vmin=vmin, vmax=vmax)
cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label(sst_celsius.long_name + f" [{sst_celsius.units}]", fontsize=12)
cbar.ax.tick_params(labelsize=10)

In [ ]:
historical = fetch.fetch_data("red_sea", "1991-01-01", "2020-12-31")
historical_monthly = (historical["analysed_sst"]
                      .mean(dim=["latitude", "longitude"])
                      .groupby("time.month").mean()
                      .compute()) + KELVIN_TO_CELSIUS

In [ ]:
# Calculate the highest mean temp per month and list the mean 
# temperatures each month for the last year
monthly_mean = (ds["analysed_sst"]
                .groupby("time.month").mean() 
                .mean(dim=["latitude", "longitude"])
                .compute()) + KELVIN_TO_CELSIUS
monthly_mean.attrs["units"] = "\u00b0C"
# Convert to DataFrame
print (monthly_mean.to_dataframe(name="SST (°C)"))

In [ ]:
df2 = pd.DataFrame({
    "2023 avg (°C)": (monthly_mean).round(2),
    "Deviatoin from 30yr mean temp (°C)": (monthly_mean - historical_monthly).round(2)
}, index=[calendar.month_name[m] for m in range(1,13)])
df2.index.name = "Month"
df2